# Pipeline Scraping Dosen Infokom UNESA v4
**Modular Pipeline** | All logic in `scraping_modules/` | Notebook = orchestrator only.

| Step | Function | Output |
|------|----------|--------|
| 1 | `run_web_step()` | `raw_web_data.csv` (Source of Truth) |
| 2 | `run_pddikti_step()` | `raw_pddikti_data.csv` (Enrichment) |
| 3 | `run_smart_merge()` | `dosen_infokom_merged.csv` (Web-First + Dedup) |
| 4 | `run_enrichment()` | `dosen_infokom_final.csv` (SimCV+Sinta+SciVal+Scholar) |
| 5 | `run_post_processing()` | `dosen_infokom_final.csv` (Clean + Final Dedup) |
| 6 | `run_supabase_sync()` | Supabase DB |

> Ensure `credentials_new.json` is present in the root folder.

In [1]:
# Setup & Imports
import sys, importlib
from pathlib import Path

ROOT_DIR = Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

# Force reload modules to pick up latest changes
for mod_name in list(sys.modules.keys()):
    if 'scraping_modules' in mod_name:
        del sys.modules[mod_name]

from scraping_modules import pipeline, config
print(f"Save Directory: {config.SAVE_DIR}")
print(f"Proxy: {'Active' if config.PROXY_URL else 'Inactive'}")
print(f"SciVal: {'Enabled' if config.ENABLE_SCIVAL else 'Disabled'}")

✅ Config: Loaded Credentials (Proxy: brd.superproxy.io:33335)
✅ Config: Loaded Credentials (Supabase: https://wfjzdhaaldwyiajbyzln.supabase.co)
Save Directory: C:\Users\rizky\OneDrive\Dokumen\GitHub\Tugas_Akhir\scraping\file_tabulars
Proxy: Active
SciVal: Enabled


### Step 1: Web Scraping (Source of Truth)

In [2]:
pipeline.run_web_step()


🌐 STARTING WEB SCRAPING...

🌐 STARTING WEB SCRAPING...
   🌍 Scraping: S1 Teknik Informatika (https://ti.ft.unesa.ac.id/page/dosen)
      ✅ Parsed: 12
   🌍 Scraping: S1 Sistem Informasi (https://si.ft.unesa.ac.id/page/dosen)
      ✅ Parsed: 13
   🌍 Scraping: S1 Pendidikan Teknologi Informasi (https://pendidikan-ti.ft.unesa.ac.id/page/dosen)
      ✅ Parsed: 16
   🌍 Scraping: S1 Teknik Elektro (https://teknikelektro.ft.unesa.ac.id/page/dosen)
      ✅ Parsed: 30
   🌍 Scraping: S1 Kecerdasan Artifisial (https://ai.fmipa.unesa.ac.id/page/dosen-s1-kecerdasan-artifisial)
      ✅ Parsed: 5
   🌍 Scraping: S1 Sains Data (https://datascience.fmipa.unesa.ac.id//page/dosen)
      ✅ Parsed: 14
   🌍 Scraping: S1 Bisnis Digital (https://bisnisdigital.feb.unesa.ac.id/page/dosen)
      ✅ Parsed: 21
   🌍 Scraping: D4 Manajemen Informatika (https://terapan-ti.vokasi.unesa.ac.id/page/dosen)
      ✅ Parsed: 14
   🌍 Scraping: S2 Informatika (https://s2if.ft.unesa.ac.id/page/dosen)
      ✅ Parsed: 5
   🌍 Scra

WindowsPath('C:/Users/rizky/OneDrive/Dokumen/GitHub/Tugas_Akhir/scraping/file_tabulars/raw_web_data.csv')

### Step 2: PDDIKTI Collection (Enrichment)

In [3]:
pipeline.run_pddikti_step()

INFO:pddiktipy.api:PDDIKTI API client initialized successfully



📡 STARTING PDDIKTI COLLECTION for 10 Active Configs...
   🔍 PDDIKTI Search: 'Teknik Informatika Universitas Negeri Surabaya'...
      ✅ Added: 9
   🔍 PDDIKTI Search: 'Sistem Informasi Universitas Negeri Surabaya'...
      ✅ Added: 10
   🔍 PDDIKTI Search: 'Pendidikan Teknologi Informasi Universitas Negeri Surabaya'...
      ✅ Added: 20
   🔍 PDDIKTI Search: 'Teknik Elektro Universitas Negeri Surabaya'...
      ✅ Added: 33
   🔍 PDDIKTI Search: 'Kecerdasan Artifisial Universitas Negeri Surabaya'...
      ✅ Added: 5
   🔍 PDDIKTI Search: 'Sains Data Universitas Negeri Surabaya'...
      ✅ Added: 14
   🔍 PDDIKTI Search: 'Bisnis Digital Universitas Negeri Surabaya'...
      ✅ Added: 20
   🔍 PDDIKTI Search: 'Manajemen Informatika Universitas Negeri Surabaya'...
      ✅ Added: 13
   🔍 PDDIKTI Search: 'Informatika Universitas Negeri Surabaya'...
      ✅ Added: 10
   🔍 PDDIKTI Search: 'Pendidikan Teknologi Informasi Universitas Negeri Surabaya'...
      ✅ Added: 0
   💾 Step 2: PDDIKTI | Saved: ra

WindowsPath('C:/Users/rizky/OneDrive/Dokumen/GitHub/Tugas_Akhir/scraping/file_tabulars/raw_pddikti_data.csv')

### Step 3: Smart Merge (Web-First + Dedup)

In [4]:
pipeline.run_smart_merge()


🔄 STARTING SMART MERGE (WEB-FIRST)...
   📊 Web Base Records: 131
   🔍 Matching PDDIKTI -> Web (exact -> NIDN -> substring -> fuzzy >=0.85)...
   🔍 Running deduplication...
   🔗 Dedup: removed 2 duplicate(s)
   💾 Step 3: Smart Merge | Saved: dosen_infokom_merged.csv (129 records)
   📊 PDDIKTI Enriched: 114 | Skipped: 20
   📊 Final Merged: 129 records


WindowsPath('C:/Users/rizky/OneDrive/Dokumen/GitHub/Tugas_Akhir/scraping/file_tabulars/dosen_infokom_merged.csv')

### Step 4: Enrichment (SimCV + Sinta + SciVal + Scholar)
Set `scholar_sample=5` to limit Scholar API calls during testing.

In [ ]:
# scholar_sample=5 for testing, set to None for full run
pipeline.run_enrichment(scholar_sample=None)


📦 STEP 4: ENRICHMENT PIPELINE
   📊 Input: 129 records from dosen_infokom_merged.csv

   [4a] Enriching NIP/NIDN from SimCV...


SimCV: 100%|██████████| 129/129 [00:53<00:00,  2.42it/s]


      Searched: 32, Skipped: 97, Enriched: 18
   💾 After SimCV | Saved: dosen_infokom_final.csv (129 records)

   [4b] Enriching Sinta IDs...
   📂 Sinta Crawl: S1 Teknik Informatika
   📂 Sinta Crawl: S2 Informatika
   📂 Sinta Crawl: S1 Pendidikan Teknologi Informasi
   📂 Sinta Crawl: S1 Sistem Informasi
   📂 Sinta Crawl: S1 Teknik Elektro
   📂 Sinta Crawl: S1 Kecerdasan Artifisial
   📂 Sinta Crawl: S1 Sains Data
   📂 Sinta Crawl: S1 Bisnis Digital
   📂 Sinta Crawl: D4 Manajemen Informatika
   📂 Sinta Crawl: S2 Pendidikan Teknologi Informasi
      Cached 105 Sinta Profiles.


Sinta: 100%|██████████| 129/129 [00:00<00:00, 15318.08it/s]
INFO:WDM:====== WebDriver manager ======


      Enriched: 94 Sinta IDs
   💾 After Sinta | Saved: dosen_infokom_final.csv (129 records)

   [4c] Running SciVal Automation...

🤖 STARTING SCIVAL AUTOMATION (Elsevier OAuth)...
   Missing Scopus IDs to find: 129


INFO:WDM:Get LATEST chromedriver version for google-chrome
INFO:WDM:Get LATEST chromedriver version for google-chrome
INFO:WDM:Get LATEST chromedriver version for google-chrome
INFO:WDM:WebDriver version 145.0.7632.77 selected
INFO:WDM:Modern chrome version https://storage.googleapis.com/chrome-for-testing-public/145.0.7632.77/win32/chromedriver-win32.zip
INFO:WDM:About to download new driver from https://storage.googleapis.com/chrome-for-testing-public/145.0.7632.77/win32/chromedriver-win32.zip
INFO:WDM:Driver downloading response is 200
INFO:WDM:Get LATEST chromedriver version for google-chrome
INFO:WDM:Driver has been saved in cache [C:\Users\rizky\.wdm\drivers\chromedriver\win64\145.0.7632.77]


   📍 Logging into SciVal...
   ✅ Credentials Submitted
   ✅ Login Success!
   📍 Navigating to UNESA Author List...
   ⬇️ Triggering CSV Download...
   ⏳ Waiting for file...
   ✅ Downloaded: All_Authors_Universitas+Negeri+Surabaya_20260220.csv
   🔄 Processing SciVal Data...
   📊 Built lookup: 3282 names
      🔄 CORRECTED: Aditya Prapanca, S.T., M.Kom. (None -> 57215981815)
      🔄 CORRECTED: Anita Qoiriah, S.Kom., M.Kom. (None -> 57201351214)
      🔄 CORRECTED: Dr. Yuni Yamasari, S.Kom., M.Kom.Dosen (None -> 57193733535)
      🔄 CORRECTED: Agus Prihanto, S.T., M.Kom. (None -> 57201350852)
      🔄 CORRECTED: Dr. Ir. Ricky Eka Putra, S.Kom., M.Kom.Dosen (None -> 56038565900)
      🔄 CORRECTED: I Made Suartana, S.Kom., M.Kom. (None -> 57189691714)
      🔄 CORRECTED: Naim Rochmawati, S.Kom., M.T. (None -> 57195258934)
      🔄 CORRECTED: Paramitha Nerisafitra, S.ST., M.Kom. (None -> 57202388830)
      🔄 CORRECTED: Ervin Yohannes, S.Kom., M.Kom., M.Sc., Ph.D.Dosen (None -> 57193503081)
      

Scholar: 100%|██████████| 5/5 [00:19<00:00,  3.82s/it]

      Verified: 5, Found New: 0
   💾 After Scholar | Saved: dosen_infokom_final.csv (129 records)

   ✅ Enrichment Complete: 129 records


WindowsPath('C:/Users/rizky/OneDrive/Dokumen/GitHub/Tugas_Akhir/scraping/file_tabulars/dosen_infokom_final.csv')

### Step 5: Post-Processing (Final Clean)

In [ ]:
pipeline.run_post_processing()


🧹 STEP 5: POST-PROCESSING...
   📊 Input: 129 records
   💾 Post-Processing | Saved: dosen_infokom_final.csv (129 records)

   📊 FINAL DATASET: 129 records
      nip: 129/129 (100.0%)
      nidn: 115/129 (89.1%)
      scholar_id: 125/129 (96.9%)
      scopus_id: 106/129 (82.2%)
      sinta_id: 94/129 (72.9%)


WindowsPath('C:/Users/rizky/OneDrive/Dokumen/GitHub/Tugas_Akhir/scraping/file_tabulars/dosen_infokom_final.csv')

### Step 6: Supabase Sync

In [ ]:
pipeline.run_supabase_sync()

### Final Result

In [7]:
import pandas as pd
from scraping_modules.config import SAVE_DIR

csv_path = SAVE_DIR / "dosen_infokom_final.csv"
if csv_path.exists():
    df = pd.read_csv(csv_path, dtype=str)
    print(f"Final Dataset: {len(df)} records")
    print(f"Columns: {list(df.columns)}")
    display(df.head(10))
else:
    print("Final CSV not found.")

Final Dataset: 129 records
Columns: ['nama_dosen', 'nama_norm', 'nip', 'nidn', 'prodi', 'scholar_id', 'scopus_id', 'sinta_id', 'source']


,nama_dosen,nama_norm,nip,nidn,prodi,scholar_id,scopus_id,sinta_id,source
0,"Aditya Prapanca, S.T., M.Kom.",Aditya Prapanca,197411012003121001,0001117406,S1 Teknik Informatika,WdhKN0sAAAAJ,57215981815,5999078,WEB+PDDIKTI
1,"Anita Qoiriah, S.Kom., M.Kom.",Anita Qoiriah,196901251995122001,0025016903,S1 Teknik Informatika,N6RJilIAAAAJ,57201351214,6010612,WEB+PDDIKTI
2,"Dr. Yuni Yamasari, S.Kom., M.Kom.Dosen",Yuni Yamasari,197506022003122001,0002067504,S2 Informatika,hn5jrnAAAAAJ,57193733535,5999992,WEB+PDDIKTI
3,"Agus Prihanto, S.T., M.Kom.",Agus Prihanto,197908062006041001,0006087903,S1 Teknik Informatika,-_VNTSIAAAAJ,57201350852,6010659,WEB+PDDIKTI
4,"Dr. Ir. Ricky Eka Putra, S.Kom., M.Kom.Dosen",Ricky Eka Putra,198701162018031001,0716018704,S2 Informatika,cPj-iOIAAAAJ,56038565900,6009054,WEB+PDDIKTI
5,"I Made Suartana, S.Kom., M.Kom.",I Made Suartana,198411242015041003,0024118405,S1 Teknik Informatika,RvG7jG4AAAAJ,57189691714,5989535,WEB+PDDIKTI
6,"Naim Rochmawati, S.Kom., M.T.",Naim Rochmawati,197512032005012001,0003127502,S1 Teknik Informatika,8wFXNjYAAAAJ,57195258934,5991531,WEB+PDDIKTI
7,"Paramitha Nerisafitra, S.ST., M.Kom.",Paramitha Nerisafitra,198905292019032013,0729058902,S1 Teknik Informatika,Em3aIV0AAAAJ,57202388830,6937234,WEB+PDDIKTI
8,"Ronggo Alit, S.Kom., M.M., M.T.",Ronggo Alit,198412082020121002,0708128403,S1 Teknik Informatika,fkxhjZ4AAAAJ,NaN,NaN,WEB
9,"Ervin Yohannes, S.Kom., M.Kom., M.Sc., Ph.D.Dosen",Ervin Yohannes,202309114,0001079106,S2 Informatika,nyyfjCgAAAAJ,57193503081,6893377,WEB+PDDIKTI
